---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-33: LangChain <b>Chains</b> Component</h1>

# Learning agenda of this notebook  

1. Overview of LangChain Chains Component
    - Example 1: Sequential Chain
    - Example 2: Sequential Chain
    - Example 3: Parallel Chain (Multiple LLM Collaboration)
    - Example 4: Conditional Chain

# <span style='background :lightgreen' >Recap: Core Components of [LangChain](https://github.com/langchain-ai/langchain)</span>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain is an open source framework that provides us with a set of tools and abstractions that make it easier for us to create complex LLM-powered applications.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Langchain components can be chained together to create sophisticated AI applications like chatbots, question-answering systems, and intelligent agents.</h3>


| Component | Description | Key Benefit |
|-----------|--------------|-------------|
| **Models** | LangChain's "universal remote control" for AI—one interface to rule them all, freeing developers from vendor-specific complexity and enabling true AI provider independence. | Write once, use with any AI model |
| **Prompts** | Reusable, parameterized templates that standardize how you communicate with AI models. | Consistent messaging with easy updates | 
| **Memory** | LangChain Memory enables AI models to maintain context across multiple interactions in a conversation. Without memory, each interaction is independent - the model has no knowledge of previous exchanges. | Natural, flowing conversations |
| **Chains** | LangChain’s “AI Assembly Lines” — they connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency. | Transform complex tasks into simple sequences | 
| **Indexes** | Indexes are LangChain's smart knowledge management systems that convert your raw documents (PDFs, websites, databases) into searchable libraries where AI models can quickly find and retrieve only the most relevant information w/o exceeding context limits. | Fast retrieval, reduced API costs | 
| **Agents** | LangChain Agents are the “decision-making brains” of your AI system — they enable models to reason, plan, and dynamically decide which tools to use and what actions to take in order to achieve a user’s goal.| Autonomous problem-solving without manual programming |

# <span style='background :lightgreen' >1. Overview of LangChain **Chains** Component</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Chains are LangChain’s “AI Assembly Lines” that connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Simply speaking, chains allows you to link the output of one model to be the input of another model call</h3>


## What Are Chains?
- Chains are structured pipelines that connect multiple components,  such as prompts, models, or data transformers — so that the output of one step becomes the input to the next.
- They let you combine multiple LLM calls (and optionally external tools or data) to perform multi-stage tasks, like "translating → summarizing → extracting → analyzing → and formatting results"
- Instead of sending one big prompt to a model, Chains let you build logical sequences that are easier to manage, debug, and reuse.
- LangChain provides both pre-built chain types and the ability to design your own custom pipelines using any mix of models, prompts, and outputs.
- This modular design encourages transparency, flexibility, and scalability.

## The Core Problem: Disconnected AI Tasks
- **Challenge:** Real-world AI applications rarely stop at a single model call. Typical workflows involve multiple reasoning steps like "translating → summarizing → extracting → analyzing → and formatting results"
- **Without LangChain:**
    - Developers manually stitch together separate LLM calls.
    - Data has to be passed around by hand between steps.
    - The code becomes repetitive, hard to debug, and difficult to scale.
    - You can’t easily reuse logic or components across different projects.
    - **Example of the Problem:**
        - You build a small app that first analyzes a document, then summarizes it.
        - You call one model for analysis.
        - You collect the result manually.
        - You call another model for summarization.
        - You handle retries, errors, and formatting on your own.
- **With LangChain:** LangChain simplifies everything by letting you chain tasks together. Think of it like this:
    - Without LangChain: You’re running two separate machines by hand — starting one, collecting its output, then feeding it into the next.
    - With LangChain: You’ve built a smooth conveyor belt — where each machine automatically hands off its result to the next stage, creating a continuous and reliable process.


# Example 1: Sequential Chain

```mermaid
graph LR
    A["📄 Input Text<br/>(English Paragraph)"]
    B["🔤 Translation Chain<br/>LLM: ChatOpenAI (llama-3.1-8b-instant)<br/>Task: English → Urdu"]
    C["📝 Output Text<br/>(Urdu Translation)"]

    A -->|Text Input| B
    B -->|Translated Urdu Text| C

    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style C fill:#f1f8e9,stroke:#7cb342,stroke-width:2px
```
> You may have to install grandalf using `uv add grandalf` command

In [1]:
import os
from langchain_openai import ChatOpenAI            
from dotenv import load_dotenv                     
from langchain_core.prompts import PromptTemplate          # For creating parameterized prompt templates
from langchain_core.output_parsers import StrOutputParser  # To cleanly extract the LLM response text

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1", temperature=0.3)

# The English paragraph to be translated
text_to_translate = """
Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the  Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. 
He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy 
inference models applied to operating systems, embedded systems, and  cloud-based services, particularly in decision-making under uncertain and 
imprecise conditions. With over 33 years of experience in teaching and  management, Dr. Butt has served in both the Pakistan Army and University 
of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, 
cybersecurity, and artificial intelligence. Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. 
In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit development, excelling in identifying and mitigating critical security risks 
across diverse platforms. His deep understanding of software architectures, memory corruption techniques, and attack vectors strengthens his ability to proactively enhance cybersecurity defences. 
A dedicated and results-driven professional, Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. 
His expertise in cybersecurity and emerging technologies enables him to contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.
"""

# Ask the user to specify the target translation language
#target_language = input("Enter the target language (e.g., Urdu, French, Spanish)")
target_language = "Urdu"

# Define the Prompt Template that will format user query into a structured prompt
prompt = PromptTemplate.from_template("Translate the following English text into {language}:\n\n{text}")

# Define an Output Parser as LLM responses can sometimes be objects, message dicts, or JSONs.
# StrOutputParser() extracts only the textual content for easy readability and ensures that the model’s response is returned as a clean, plain string.
parser = StrOutputParser()
#parser = JsonOutputParser()



# Build the LangChain Expression Language (LCEL) Chain
# The | (pipe) operator connects multiple components sequentially into a runnable “chain”:
      #   prompt      → Renders the text prompt with user-specified values
      #   llama_model → Sends the rendered prompt to the model which automatically calls its  invoke() method
      #   parser      → Extracts and formats the final text output for display
chain = prompt | llama_model | parser


# Run the Chain by calling its .invoke() method and providing it the required inputs
     #   Passes user input into the prompt and generate the final prompt, i.e., "Translate the following English text into {language}:\n\n{text}'
     #   Automatically forwards the rendered prompt to the LLM
     #   Finally the parser automatically calls its parse() method and returns the model’s final translated text as a string
result = chain.invoke({'language': target_language, 'text': text_to_translate})
print(result)

# Every LCEL chain internally forms a computation graph (nodes = components, edges = data flow).
chain.get_graph().print_ascii()  # prints a simple ASCII visualization of the pipeline

ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس ، پنجاب یونیورسٹی (پی یو) ، لاہور ، پاکستان میں ایک ممتاز اسسٹنٹ پروفیسر ہیں۔ 
وہ پی یو سی آئی ٹی ، پنجاب یونیورسٹی سے کمپیوٹر سائنس میں ایم ایس سی اور ایم فل (دونوں سونے کے تمغے کے ساتھ) اور پی ایچ ڈی کی ڈگری رکھتے ہیں۔ ان کی تحقیق غیر یقینی اور غیر واضح حالات میں فیصلہ سازی میں خصوصاً آپریٹنگ سسٹم ، امبیڈڈ سسٹم اور کلاؤڈ بیسڈ سروسز پر لاگو کی گئی فزzy انفرنس ماڈلز پر مرکوز ہے۔ 
تعلیم اور انتظامیہ میں 33 سال سے زائد کے تجربے کے ساتھ ، ڈاکٹر بٹ نے پاک فوج اور پنجاب یونیورسٹی دونوں میں خدمات انجام دی ہیں ، جو بین الاختصاصی مہارت کا خزانہ لاتے ہیں۔ ان کی تدریسی مہارتوں میں امبیڈڈ اور ریئل ٹائم آپریٹنگ سسٹم ، سسٹم پروگرامنگ ، سائبر سیکیورٹی اور آرٹیفیشل انٹیلی جنس شامل ہیں۔ 
اکادمی کے علاوہ ، وہ ایک ٹیکنالوجی کے کاروباری ، ایکسکالیٹ اور فالکن ہنٹ کے بانی اور ٹی بکس سلوشنز کے شریک بانی کے طور پر خدمات انجام دے رہے ہیں۔ 
حالیہ برسوں میں ، انہوں نے کمزوری کی تحقیق ، بائنری منافع اور منافع کی ترقی میں نمایاں مہارت حاصل کی ہے ، اور متنوع پلیٹ فارمز پر

# Example 2: Sequential Chain
```mermaid
graph LR
    A["📄 Input Document<br/>(1000-word English text)"]
    B["🔤 Translation Chain<br/>LLM: ChatOpenAI (gpt-4o-mini)<br/>Task: English → Urdu"]
    C["📝 Urdu Translation Output"]
    D["🧠 Summarization Chain<br/>LLM: ChatOpenAI (gpt-4o-mini)<br/>Task: Summarize to &lt;100 words"]
    E["📚 Final Output<br/>(Concise Urdu Summary)"]

    A -->|Text Input| B
    B -->|Translated Urdu Text| C
    C -->|Passes as Input| D
    D -->|Summarized Urdu Text| E

    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style C fill:#f1f8e9,stroke:#7cb342,stroke-width:2px
    style D fill:#ede7f6,stroke:#673ab7,stroke-width:2px
    style E fill:#fce4ec,stroke:#c2185b,stroke-width:2px
```
### ===== Description of `RunnableLambda` Method Used in following Code =====
- **What is `RunnableLambda()`?**
    - A wrapper that turns any plain Python function (including lambdas) into a LangChain-compatible Runnable
    - Makes arbitrary transformation logic a first-class citizen in a LangChain pipeline
    - Think of it as an adapter — it bridges the gap between "raw Python logic" and "chain-ready component"
- **Why use `RunnableLambda()`?**
    - LangChain chains expect Runnables, not bare functions — RunnableLambda makes your function chainable via |
    - Lets you reshape, reformat, or enrich data between steps in a pipeline without building a full custom class
    - Gives you full Python flexibility at any point in the chain
>  **Without `RunnableLambda`, the chain would receive a raw string and the prompt template would have no way to unpack it into its required variables.** 

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize model
model1 = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model2 = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1", temperature=0.3)

# Input data
target_language = "Urdu"
# The English paragraph to be translated
text_to_translate = """
Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the  Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. 
He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy 
inference models applied to operating systems, embedded systems, and  cloud-based services, particularly in decision-making under uncertain and 
imprecise conditions. With over 33 years of experience in teaching and  management, Dr. Butt has served in both the Pakistan Army and University 
of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, 
cybersecurity, and artificial intelligence. Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. 
In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit development, excelling in identifying and mitigating critical security risks 
across diverse platforms. His deep understanding of software architectures, memory corruption techniques, and attack vectors strengthens his ability to proactively enhance cybersecurity defences. 
A dedicated and results-driven professional, Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. 
His expertise in cybersecurity and emerging technologies enables him to contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.
"""

# Prompts
prompt_translate = PromptTemplate.from_template("Translate the following English text into {language}:\n\n{text}")
prompt_summarize = PromptTemplate.from_template("Summarize the following {language} text in less than 50 words:\n\n{text}")


# Parser
parser = StrOutputParser()

# RunnableLambda to pass translation as structured input (Uses RunnableLambda to transform the output between stages)
format_input = RunnableLambda(lambda text: {"language": target_language, "text": text})

# Build final chain and run it
chain = prompt_translate | model1 | parser | format_input | prompt_summarize | model2 | parser
result = chain.invoke({"language": target_language, "text": text_to_translate})

print("\n===== Final Concise Summary in", target_language, "=====\n")
print(result)
chain.get_graph().print_ascii()  # prints a simple ASCII visualization of the pipeline


===== Final Concise Summary in Urdu =====

ڈاکٹر محمد عارف بٹ پنجاب یونیورسٹی میں معزز اسسٹنٹ پروفیسر ہیں، جو فیوزے انفرنس ماڈلز اور سائبر سیکیورٹی پر تحقیق کرتے ہیں۔
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *       

#  Example 3: Parallel Chain (Multiple LLM Collaboration)
- This example demonstrates how to:
    - 1️⃣ Send the same user prompt to multiple LLMs (running in parallel)
    - 2️⃣ Collect their independent outputs
    - 3️⃣ Pass both results into a third model that synthesizes a unified summary


```mermaid
graph LR
    %% Nodes
    A["🧑‍💻 User Prompt<br/>'Generate a detailed report on the 9/11 incident'"]
    B["🤖 LLM-A<br/>Model: GPT-4<br/>Task: Generate Report (Perspective 1)"]
    C["🤖 LLM-B<br/>Model: Claude 3<br/>Task: Generate Report (Perspective 2)"]
    D["🧠 LLM-C<br/>Model: Gemini-Pro<br/>Task: Combine & Synthesize<br/>both reports into a unified summary"]
    E["📘 Final Output<br/>Unified 9/11 Report"]

    %% Connections
    A -->|Same Prompt| B
    A -->|Same Prompt| C
    B -->|Report A| D
    C -->|Report B| D
    D -->|Merged Report| E

    %% Styling
    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style C fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style D fill:#ede7f6,stroke:#673ab7,stroke-width:2px
    style E fill:#fce4ec,stroke:#c2185b,stroke-width:2px
```


### ===== Description of New Methods Used in following Code =====

- **`.partial()`: STRING EXTRACTION COMPONENT**
    - What is `.partial()`?
        - A method that "pre-fills" or "locks in" some variables in a prompt template
        - Creates a new prompt template with fewer required input variables
        - Think of it as "partial application" from functional programming
    - Why use `.partial()`?
        - Allows you to create specialized versions of a generic template
        - Reduces the number of variables you need to pass at runtime
        - Enables cleaner parallel execution with different fixed parameters
    - Example:
        - Original template: "Write about {topic} from {perspective}"
        - After .partial(perspective="American"): "Write about {topic} from American perspective"
        - Now only needs: {topic}
    - How it works in our code?
        - We have ONE template: prompt_report (needs topic + perspective)
        - We use `.partial()` to create TWO specialized versions:
            - `chain_american`: perspective locked to "the American perspective"
            - `chain_middle_eastern`: perspective locked to "the Middle Eastern perspective"
        - Each chain now only needs {topic} at runtime
    - This allows RunnableParallel to pass the same input to both chains
- **`RunnableParallel()`: CONCURRENT EXECUTION ORCHESTRATOR**
   - Executes multiple chains/runnables concurrently
   - Takes a dictionary, where
       -  Keys become the output variable names
       -  Values are the chains/runnables to execute
   - Passes same input to all chains
   - Returns dictionary will all results: {key: result}
   - Faster than sequential execution
   - Parallel execution means order is not guaranteed (but doesn't matter here)

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

load_dotenv("../keys/.env", override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize three different or same LLMs (I am using one model using the same Groq endpoint for simplicity), you could use separate ones for realism
# First perspective generator (acts as GPT-4 equivalent)
llm_a = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
# Second perspective generator (acts as Claude equivalent)
llm_b = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
# Synthesizer/combiner model (acts as Gemini equivalent)
llm_c = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


# ==================== PROMPT TEMPLATES ====================
# Define the base prompt template that will be used by both LLM-A and LLM-B. This template requires TWO variables: {topic} and {perspective}
prompt_report = PromptTemplate.from_template("Write a detailed analytical report about the {topic} from {perspective}.")

# Define the synthesis prompt for LLM-C (the combiner). This template takes the outputs from both parallel chains and combines them into a unified summary
# It expects TWO variables: {report_a} and {report_b} 
prompt_synth = PromptTemplate(
                                template="""Combine the following two reports into a single, cohesive summary.
                                Report A:{report_a}
                                Report B:{report_b}
                                Synthesize both perspectives into one unified report in less than 50 words.""",
                                input_variables=["report_a", "report_b"]
                            )


# ==================== OUTPUT PARSER ====================
# StrOutputParser() extracts string content from LLM response objects and converts AIMessage → plain string. Essential for chaining LLM outputs to prompts
parser = StrOutputParser()

# ==================== .partial() METHOD  ====================
# Create first specialized chain. The .partial() locks in perspective = "the American perspective", so this chain only needs topic as input
# With pre-fill perspective, llm_a will generate report from the American perspective and extract string from AIMessage object
chain_american = (  prompt_report.partial(perspective="the American perspective")   |    llm_a      |    parser  )

# Create second specialized chain. The .partial() locks in perspective = "the Middle Eastern perspective", so this chain only needs topic as input
# With pre-fill perspective, llm_a will generate report from the American perspective and extract string from AIMessage object
chain_middle_eastern = (  prompt_report.partial(perspective="the Middle Eastern perspective")    |   llm_b     |    parser   )

# ==================== PARALLEL EXECUTION ====================
# INPUT/OUTPUT FLOW:
# Input:  {"topic": "9/11 incident"}
# ↓
# RunnableParallel splits input to both chains simultaneously:
# ├─→ chain_american receives {"topic": "9/11 incident"}
# │   └─→ Generates report from American perspective
# └─→ chain_middle_eastern receives {"topic": "9/11 incident"}
#     └─→ Generates report from Middle Eastern perspective
# ↓
# Output: {
#   "report_a": "American perspective analysis...",
#   "report_b": "Middle Eastern perspective analysis..."
# }
parallel_chain = RunnableParallel({
    # Key "report_a" will be used in prompt_synth as {report_a}
    "report_a": chain_american,
    
    # Key "report_b" will be used in prompt_synth as {report_b}
    "report_b": chain_middle_eastern
})


# ==================== SYNTHESIS CHAIN ====================
# Sequential chain that combines the parallel outputs
# Takes the dictionary output from parallel_chain and synthesizes it
# FLOW:
# Input: {"report_a": "text1", "report_b": "text2"}
# ↓
# prompt_synth: Formats the synthesis prompt with both reports
# ↓
# llm_c: Generates unified summary combining both perspectives
# ↓
# parser: Extracts final string output
# ↓
# Output: "Unified synthesized report text..."
merge_chain = prompt_synth | llm_c | parser

# ==================== FINAL CHAIN PIPELINE ====================
# COMPLETE EXECUTION FLOW:
#
# Step 1: User provides input
#         {"topic": "9/11 incident"}
#
# Step 2: parallel_chain executes (CONCURRENT)
#         ├─→ chain_american generates report_a
#         └─→ chain_middle_eastern generates report_b
#         Output: {"report_a": "...", "report_b": "..."}
#
# Step 3: merge_chain executes (SEQUENTIAL)
#         prompt_synth formats both reports
#         ↓
#         llm_c synthesizes into unified report
#         ↓
#         parser extracts final string
#
# Step 4: Final output
#         "Synthesized unified report text..."
#
# The | operator chains these components sequentially:
# parallel_chain output → merge_chain input
final_chain = parallel_chain | merge_chain

# ==================== EXECUTION ====================
# Run the complete chain pipeline
# Note: We only need to provide 'topic' now because 'perspective' 
# has been pre-filled using .partial() in each specialized chain
user_prompt = {"topic": "9/11 incident"}

print("\n===== 🔄 EXECUTING PARALLEL CHAINS =====\n")
print(f"Topic: {user_prompt['topic']}")
print("Generating reports from multiple perspectives simultaneously...")
print()

# Invoke the entire chain
# This triggers the full pipeline: parallel generation → synthesis
result = final_chain.invoke(user_prompt)

print("\n===== 🧾 FINAL UNIFIED REPORT =====\n")
print(result)
print()

print("\n===== 📊 CHAIN EXECUTION GRAPH =====\n")
final_chain.get_graph().print_ascii()


===== 🔄 EXECUTING PARALLEL CHAINS =====

Topic: 9/11 incident
Generating reports from multiple perspectives simultaneously...


===== 🧾 FINAL UNIFIED REPORT =====

The 9/11 attacks had far-reaching consequences, impacting US foreign policy, Middle Eastern relations, and global security, with ongoing effects on international relations, terrorism, and cultural exchange.


===== 📊 CHAIN EXECUTION GRAPH =====

        +----------------------------------+         
        | Parallel<report_a,report_b>Input |         
        +----------------------------------+         
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *      

# Example 4: Conditional Chain
- A Customer Feedback Processing Pipeline — where the system first analyzes sentiment of user feedback, and routes the text to different LLMs based on whether the feedback is positive or negative.
- **Scenario:** Customer Feedback Analyzer (Conditional Chain)
    - Step 1: User submits feedback (text input).
    - Step 2: A sentiment analysis LLM classifies it as positive or negative.
    - Step 3 (Branch A): If positive, send to LLM-A for thank-you message generation.
    - Step 3 (Branch B): If negative, send to LLM-B for apology & escalation message generation.
    - Step 4: Output the final response.
  
```mermaid
graph LR
    %% Nodes
    A["💬 <b>Customer Feedback</b><br/><br/>Input: Text from user (positive or negative)"]
    B["🧠 <b>Sentiment Analysis LLM</b><br/><br/>Task: Classify feedback<br/>as Positive or Negative"]
    C["😊 <b>Positive Feedback Chain</b><br/><br/>LLM-A:<br/>Generate a friendly thank-you message"]
    D["😞 <b>Negative Feedback Chain</b><br/><br/>LLM-B:<br/>Generate an apology and escalate issue"]
    E["📤 <b>Final Output</b><br/><br/>Personalized AI response<br/>based on detected sentiment"]

    %% Connections
    A --> B
    B -->|Positive Feedback| C
    B -->|Negative Feedback| D
    C --> E
    D --> E

    %% Styling
    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px,color:#0d47a1,font-size:14px
    style B fill:#ede7f6,stroke:#673ab7,stroke-width:2px,color:#311b92,font-size:14px
    style C fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20,font-size:14px
    style D fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#b71c1c,font-size:14px
    style E fill:#fff3e0,stroke:#f57c00,stroke-width:2px,color:#e65100,font-size:14px
```

In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

# Load API Keys from environment file
load_dotenv("../keys/.env", override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize three different or same LLMs (I am using one model using the same Groq endpoint for simplicity), you could use separate ones for realism
llm_sentiment = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
llm_positive = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
llm_negative = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

parser = StrOutputParser()

# ==================== PROMPT TEMPLATES ====================
# --- (A) Sentiment Analysis Prompt ---
sentiment_prompt = PromptTemplate.from_template("Analyze the sentiment of the following feedback. Respond with ONLY one word: 'positive' or 'negative'. Feedback: {feedback}")
# --- (B) Positive Feedback Prompt ---
positive_prompt = PromptTemplate.from_template("The following feedback was positive: {feedback}. Generate a friendly thank-you message to the customer, appreciating their kind words and expressing gratitude.")
# --- (C) Negative Feedback Prompt ---
negative_prompt = PromptTemplate.from_template("The following feedback was negative: {feedback}. Generate a polite apology and inform the customer that their issue will be escalated to the support team.")

# ==================== Define Base Chains ====================
# Sentiment analysis chain
sentiment_chain = sentiment_prompt | llm_sentiment | parser
# Positive branch chain
positive_chain = positive_prompt | llm_positive | parser
# Negative branch chain
negative_chain = negative_prompt | llm_negative | parser


# ==================== Define Conditional Router ====================
# RunnableBranch lets us define if-else logic using lambda conditions
conditional_chain = RunnableBranch(
                                    (lambda output: "positive" in output.lower(), positive_chain), # Each tuple = (condition, branch_chain)
                                    (lambda output: "negative" in output.lower(), negative_chain),
                                    RunnablePassthrough()    # Default branch (if none of the above match)
                                    )

# ==================== Compose Full Chain ====================
# sentiment_chain classifies feedback, while the conditional_chain routes to the right branch
final_chain = sentiment_chain | conditional_chain

# ==================== Run the Pipeline ====================
# Try different feedback examples to see routing in action
feedback = input("Enter customer feedback: ")

result = final_chain.invoke({"feedback": feedback})

print("\n===== 🤖 AI Response =====\n")
print(result)

print("\n===== 📊 CHAIN EXECUTION GRAPH =====\n")
final_chain.get_graph().print_ascii()

Enter customer feedback:  The product stopped working in the first week, and i want to return it



===== 🤖 AI Response =====

Dear valued customer,

I am so sorry to hear that you had a negative experience. I want to start by apologizing for any inconvenience or frustration this has caused. Please know that we take all feedback seriously and are committed to making things right.

I would like to inform you that I am escalating your issue to our dedicated support team, who will work closely with you to resolve the matter as soon as possible. They will be in touch with you promptly to discuss the details and provide a suitable solution.

If there's anything I can do in the meantime to assist you, please don't hesitate to reach out. Your satisfaction is our top priority, and we appreciate your patience and cooperation as we work to resolve this issue.

Thank you for bringing this to our attention, and we look forward to the opportunity to serve you better in the future.

Best regards,
[Your Name]

===== 📊 CHAIN EXECUTION GRAPH =====

  +-------------+    
  | PromptInput |    
  +----